<h1 style="color:red;">ETL Pipeline</h1>
<hr/>
<p>Small Demo using ETL Pipeline</p>
<h3>Loading Required Library</h3>

In [21]:
import os
import pandas as pd
from openpyxl import load_workbook
from pyxlsb import open_workbook
import re

<h3>Setting Regular Expressions</h3>

In [22]:
#Month Map
month_map = {
    'jan':1,'january':1,
    'feb':2,'february':2,
    'mar':3,'march':3,
    'apr':4,'april':4,
    'may':5,
    'jun':6,'june':6,
    'jul':7,'july':7,
    'aug':8,'august':8,
    'sep':9,'sept':9,'september':9,
    'oct':10,'october':10,
    'nov':11,'november':11,
    'dec':12,'december':12
}


# Reg expression patterns
# Year standalone
year_pattern = re.compile(r'(20\d{2})')

# Full date (DD-MM-YYYY or DD/MM/YYYY)
full_date_pattern = re.compile(r'(\d{2})[-/](\d{2})[-/](20\d{2})')

# YYYY-MM or YYYY.MM or YYYY_MM
yyyy_mm_pattern = re.compile(r'(20\d{2})[._-](0[1-9]|1[0-2])')

# MM-YYYY or MM_YYYY
mm_yyyy_pattern = re.compile(r'(0[1-9]|1[0-2])[._-](20\d{2})')

# FY24 type
fy_pattern = re.compile(r'fy(\d{2})', re.IGNORECASE)

# Month Word
month_word_pattern = re.compile(
    r'(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec|'
    r'january|february|march|april|june|july|august|september|october|november|december)',
    re.IGNORECASE
)

<h3>Month-Year Extraction</h3>

In [23]:
def extract_year_month(filename):
    name = filename.lower()
    match = full_date_pattern.search(name)
    # FULL DATE
    if match:
        day, month, year = match.groups()
        return int(year), int(month)
        
    # YYYY-MM
    # -------------------------
    match = yyyy_mm_pattern.search(name)
    if match:
        year, month = match.groups()
        return int(year), int(month)
        
    # MM-YYYY
    # -------------------------
    match = mm_yyyy_pattern.search(name)
    if match:
        month, year = match.groups()
        return int(year), int(month)
        
    # FY + Month Word
    # -------------------------
    fy_match = fy_pattern.search(name)
    month_match = month_word_pattern.search(name)
    if fy_match and month_match:
        fy_year = int(fy_match.group(1))
        year = 2000 + fy_year
        month = month_map[month_match.group(0)]
        return year, month

    # Month Word + Year
    # -------------------------
    if month_match:
        month = month_map[month_match.group(0)]
        year_match = year_pattern.search(name)
        if year_match:
            year = int(year_match.group(0))
            return year, month

    return None, None

<h3>Extracting Sheets</h3>

In [24]:
def get_sheets(file_path, extension):
    sheets = []
    try:
        if extension in [".xlsx", ".xls"]:
            wb = load_workbook(file_path, read_only= True)
            sheets = wb.sheetnames
        elif extension == ".xlsb":
            with open_workbook(file_path) as wb:
                sheets = wb.sheets
        elif extension == ".csv":
            sheets = ["CSV_SINGLE_SHEET"]
    except Extension as e:
        sheets = [f"ERROR: {str(e)}"]
    return sheets

<h3>Reading Files</h3>

In [25]:
month_name_map = {
    1:"JAN",2:"FEB",3:"MAR",4:"APR",5:"MAY",6:"JUN",
    7:"JUL",8:"AUG",9:"SEP",10:"OCT",11:"NOV",12:"DEC"
}

records = []
file_count = 0

ROOT_FOLDER = r"C:\Berry\RootFolder"

system_names = [
    name for name in os.listdir(ROOT_FOLDER)
    if os.path.isdir(os.path.join(ROOT_FOLDER, name))
]
for root, dirs, files in os.walk(ROOT_FOLDER):
    for file in files:
        file_count+=1
        if file.lower().endswith(('.csv', '.xlsx', '.xlsb', '.xls')):
            full_path = os.path.join(root, file)
            extension = os.path.splitext(file)[1].lower()
            parts = full_path.split(os.sep)
            try:
                system = parts[-3]
                year_folder = parts[-2]
            except:
                system = None
                year_folder = None
            extracted_year, extracted_month = extract_year_month(file)
            if extracted_month:
                month_str = f"{extracted_month:02d}"
                month_name = month_name_map[extracted_month]
            else:
                month_str = "NA"
                month_name = "NA"
            month_str = f"{extracted_month:02d}" if extracted_month else "NA"
            year_str = str(extracted_year) if extracted_year else "NA"
            sheet_list = get_sheets(full_path, extension)
            #print(f"{system:<12} : {file:<45}  -> {month_name} {year_str}")
            print(f"{system:<12} : {file:<45} [Sheet Count: {len(sheet_list)}] -> {month_name} {year_str}", sheet_list)
        records.append({
            "System": system,
            "FileName": file,
            "Month ": month_name,
            "Year": year_str,
            "Sheet Name": sheet_list
        })
df = pd.DataFrame(records)
df.to_csv("MetaData.csv", index=False)

System_A     : 2021_Jun_ALPHA_dump.xlsx                      [Sheet Count: 4] -> JUN 2021 ['README', 'Posting ', 'NOTES', 'Pivot']
System_A     : 2021_Oct_ALPHA_dump.xlsx                      [Sheet Count: 4] -> OCT 2021 ['README', 'Posting', 'NOTES', 'Pivot']
System_A     : ALPHA FY21 April Extract FINAL.xlsx           [Sheet Count: 4] -> APR 2021 ['README', 'Posting', 'NOTES', 'Pivot']
System_A     : ALPHA FY21 August Extract FINAL.xlsx          [Sheet Count: 4] -> AUG 2021 ['README', 'MAIN EXPORT ', 'NOTES', 'Pivot']
System_A     : ALPHA FY21 December Extract FINAL.xlsx        [Sheet Count: 4] -> DEC 2021 ['README', 'Main Export ', 'NOTES', 'Pivot']
System_A     : ALPHA-2021-03 raw.xlsx                        [Sheet Count: 4] -> MAR 2021 ['README', 'Posting', 'NOTES', 'Pivot']
System_A     : ALPHA-2021-09 raw.xlsx                        [Sheet Count: 4] -> SEP 2021 ['README', 'Transactions', 'NOTES', 'Pivot']
System_A     : ALPHA__Feb-2021__export (draft).xlsx          [Sheet Count: